# 분자 생성 hands-on — RDKit + PyTorch

**생성모델 트랙 · 4교시 실습 노트북** · 실습 저장소: `github.com/fourmodern/2025_aidrugdiscovery`

> 위에서 아래로 셀을 순서대로 실행하세요 (Colab: `런타임 > 모두 실행`). **GPU 없이 CPU로도** 몇 분 내에 돌아가는 **소규모 개념 데모**입니다.

## 학습 목표
1. 소규모 **SMILES** 데이터를 불러와 유효성 검사·정제한다.
2. **char-level 언어모델(char-RNN)** 로 새로운 분자(SMILES)를 생성한다 (+ SELFIES / VAE 개념).
3. 생성물을 **validity · uniqueness · novelty** 로 평가한다.
4. **QED · SA · Lipinski** 물성 지표로 유망 후보를 필터·랭킹한다.
5. 후보를 **시각화**하고 docking/ADMET·조건부 생성 연계와 **한계**를 이해한다.

## 전체 파이프라인 (한눈에)
| 단계 | 내용 |
|---|---|
| **① 데이터** | SMILES 로딩 · 유효성 · 정제 |
| **② 생성** | char-RNN / VAE 로 새 SMILES |
| **③ 평가** | validity · uniqueness · novelty |
| **④ 필터** | QED · SA · 물성 기준 |
| **⑤ 후속** | 시각화 → docking/ADMET → 검증 |

### 한 줄 이론
- **언어모델(LM)**: 분자를 SMILES 문자열로 보고 "다음 문자"를 확률적으로 이어 붙여 새 분자를 만든다 (autoregressive).
- **VAE**: 분자를 연속 잠재공간(latent space)으로 인코딩한 뒤, 그 공간에서 샘플링·디코딩해 새 분자를 만든다.

> ⚠️ **주의**: 이 노트북의 모든 in silico 점수(QED·SA 등)는 **가설(우선순위 제안)** 이지 사실이 아닙니다. 소규모 데모라 생성 품질은 낮습니다 — 목적은 **개념 확인**입니다.


In [ ]:
# --- 설치 (Colab 기준) ---------------------------------------------------
# rdkit, selfies 는 pip 로 설치. torch 는 Colab 에 기본 설치되어 있으면 재설치하지 않음.
# 로컬(Jupyter)에서 이미 설치돼 있으면 이 셀은 빠르게 넘어갑니다.
import importlib, subprocess, sys

def _ensure(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
        print(f"[ok] {import_name} 이미 설치됨")
    except ImportError:
        print(f"[install] {pip_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

_ensure("rdkit", "rdkit")
_ensure("selfies", "selfies")
try:
    import torch  # Colab 기본 제공
    print(f"[ok] torch {torch.__version__}")
except ImportError:
    _ensure("torch", "torch")


In [ ]:
# --- 임포트 & 재현성 seed 고정 ------------------------------------------
import os, random, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, Descriptors, QED, Crippen, Lipinski
RDLogger.DisableLog("rdApp.*")   # 파싱 경고 숨김 (유효성은 우리가 직접 판정)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch:", torch.__version__)


## ① 데이터 — SMILES 로딩 · 유효성 · 정제

소규모 **drug-like** SMILES 집합을 준비합니다. 강의 데모라 **노트북에 직접 임베드한 ~150개 유효 분자**를 기본으로 쓰고,
(옵션) ChEMBL/ZINC 다운로드를 시도하되 실패해도 임베드 데이터로 그대로 진행합니다.

정제 절차: RDKit 파싱 → **canonical SMILES** 통일 → 중복 제거 → 무거운 원자 수(heavy atom) 길이 제한.


In [ ]:
# --- (기본) 노트북에 임베드된 drug-like SMILES (RDKit 로 사전 검증된 canonical 형태) ---
EMBEDDED_SMILES = [
    "CC(=O)Oc1ccccc1C(=O)O",
    "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
    "CC(C)NCC(O)COc1cccc2ccccc12",
    "Cn1c(=O)c2c(ncn2C)n(C)c1=O",
    "CN1CCC[C@H]1c1cccnc1",
    "CC(=O)Nc1ccc(O)cc1",
    "CCN(CC)CCNC(=O)c1cc(Cl)c(N)cc1OC",
    "O=C(O)c1ccccc1O",
    "CC(C)(C)NCC(O)c1ccc(O)c(CO)c1",
    "CN1CCC23c4c5ccc(O)c4OC2C(O)=CCC3C1C5",
    "COc1ccc2cc(C(C)C(=O)O)ccc2c1",
    "CC(C)Cc1ccccc1",
    "NS(=O)(=O)c1ccc(NC(=O)c2ccccc2Cl)cc1",
    "CCOC(=O)c1ccccc1",
    "O=C(O)Cc1ccccc1",
    "CC(=O)Nc1ccc(OCC(O)CNC(C)C)cc1",
    "CCN(CC)C(=O)c1ccccc1",
    "Nc1ccc(S(N)(=O)=O)cc1",
    "Cc1ncsc1C(=O)Nc1ccccc1",
    "CN(C)CCCN1c2ccccc2Sc2ccccc21",
    "CC(C)(C)c1ccc(O)cc1",
    "OCC(O)C(O)C(O)C(O)CO",
    "CC(N)Cc1ccccc1",
    "CNC(C)Cc1ccccc1",
    "COC(=O)c1ccccc1OC(C)=O",
    "COc1ccccc1OC",
    "c1ccc2[nH]ccc2c1",
    "c1ccc2ncccc2c1",
    "O=C1CCCCC1",
    "CC1CCCCC1",
    "CCOc1ccccc1",
    "Cc1ccccc1C",
    "CC(=O)c1ccccc1",
    "O=Cc1ccccc1",
    "O=C(O)c1ccc(O)cc1",
    "Nc1ccccc1",
    "Clc1ccccc1",
    "Fc1ccccc1",
    "Brc1ccccc1",
    "c1ccc(-c2ccccc2)cc1",
    "O=C(Nc1ccccc1)c1ccccc1",
    "CC(C)(C)OC(=O)N1CCNCC1",
    "O=C(O)C1CCNCC1",
    "c1ccc(CN2CCNCC2)cc1",
    "CN1CCN(C)CC1",
    "O=S(=O)(c1ccccc1)N1CCCCC1",
    "CC(=O)N1CCCCC1",
    "c1ccc(N2CCOCC2)cc1",
    "O=C1CCC(=O)N1",
    "O=C1CCCCN1",
    "COC(=O)c1ccc(N)cc1",
    "NC(=O)c1ccccc1",
    "O=C(O)C(=O)O",
    "CC(O)C(=O)O",
    "O=C(O)CCC(=O)O",
    "CCCCCCCCCCCCCCCC(=O)O",
    "CCCCCCCC(=O)O",
    "OCc1ccccc1",
    "O=S(=O)(O)c1ccccc1",
    "CN(C)c1ccccc1",
    "O=[N+]([O-])c1ccccc1",
    "Nc1ccc(C(=O)O)cc1",
    "CCOC(=O)c1ccc(N)cc1",
    "CCN(CC)C(=O)Cn1ccnc1",
    "c1ccc(Oc2ccccc2)cc1",
    "Cc1ccc(S(N)(=O)=O)cc1",
    "Cc1ccc(C(=O)O)cc1",
    "COc1ccc(C=O)cc1",
    "COc1ccc(CCN)cc1",
    "NCCc1ccc(O)c(O)c1",
    "NCCc1ccc(O)cc1",
    "NCCc1c[nH]c2ccccc12",
    "NCCc1ccccc1",
    "CC(C)(C)NCC(O)c1ccc(O)c(O)c1",
    "CC(N)Cc1ccc(O)cc1",
    "CN1CCCC1=O",
    "c1ccc2c(c1)OCO2",
    "c1ccc2ccccc2c1",
    "c1ccc2c(c1)ccc1ccccc12",
    "O=C(O)c1ccccc1C(=O)O",
    "O=C1OC(=O)c2ccccc21",
    "Cc1ccccc1N",
    "Cc1ccc(N)cc1",
    "Nc1ccc(N)cc1",
    "Oc1ccc(O)cc1",
    "Oc1ccccc1O",
    "Oc1cccc(O)c1",
    "COc1ccc(O)cc1",
    "CC(C)c1ccccc1",
    "CCc1ccccc1",
    "Cc1ccccc1",
    "NC(CO)CO",
    "Cc1cc(=O)[nH]c(=O)[nH]1",
    "O=c1cc[nH]c(=O)[nH]1",
    "Nc1ncnc2[nH]cnc12",
    "Nc1nc2[nH]cnc2c(=O)[nH]1",
    "O=c1[nH]cnc2[nH]cnc12",
    "Cn1cnc2c1c(=O)[nH]c(=O)n2C",
    "Cn1c(=O)c2[nH]cnc2n(C)c1=O",
    "OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O",
    "OC[C@H]1O[C@@](O)(CO)[C@@H](O)[C@@H]1O",
    "O=C(c1ccccc1)c1ccccc1",
    "O=C(c1ccccc1)c1ccccc1O",
    "COc1cc2c(cc1OC)C(=O)C(Cc1ccccc1)CC2",
    "O=C1c2ccccc2C(=O)c2ccccc21",
    "Clc1ccc(C(c2ccccc2)N2CCCCC2)cc1",
    "CN(C)CCc1c[nH]c2ccc(CS(N)(=O)=O)cc12",
    "CC(C)NCC(O)c1ccc(O)c(O)c1",
    "COc1ccc2[nH]cc(CCN)c2c1",
    "COc1cc2c(cc1OC)CN(C)CC2",
    "NC(=O)Cn1cncn1",
    "c1cnc2[nH]ccc2c1",
    "c1ccc2c(c1)[nH]c1ccccc12",
    "O=C1Cc2ccccc2N1",
    "O=C1CCc2ccccc21",
    "O=C1CCCc2ccccc21",
    "OC1CCCc2ccccc21",
    "c1ccc2c(c1)CCCC2",
    "c1ccc2c(c1)CCC2",
    "CC1(C)CCCCC1",
    "O=C(O)CCc1ccccc1",
    "O=C(O)/C=C/c1ccccc1",
    "OC/C=C/c1ccccc1",
    "COC(=O)c1ccccc1",
    "CCCCOC(=O)c1ccccc1",
    "NC(=O)c1cccnc1",
    "O=C(O)c1cccnc1",
    "O=C(O)c1ccncc1",
    "NC(=O)c1ccncc1",
    "Cn1ccc(=O)[nH]c1=O",
    "Cc1ncc(CO)c(CO)c1O",
    "CCOc1ccc(NC(C)=O)cc1",
    "CCN(CC)CC(=O)Nc1c(C)cccc1C",
    "COc1ccccc1C(=O)O",
    "COc1cccc(C(=O)O)c1",
    "COc1ccc(C(=O)O)cc1",
    "O=C(Nc1ccccc1)c1ccccc1O",
    "O=C(O)c1ccc(C(=O)O)cc1",
    "O=C(O)c1cccc(C(=O)O)c1",
    "Cc1cccc(C)c1",
    "Cc1ccc(C)cc1",
    "Nc1cccc(N)c1",
    "Nc1ccccc1N",
    "CC(C)(c1ccc(O)cc1)c1ccc(O)cc1",
    "CC(=O)CC(C)=O",
    "Nc1ccc(C(=O)O)c(O)c1",
    "CCOC(=O)CC(=O)OCC",
    "COc1ccc(CC(N)C(=O)O)cc1",
    "N[C@@H](Cc1ccccc1)C(=O)O",
    "N[C@@H](Cc1ccc(O)cc1)C(=O)O",
    "N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O",
    "N[C@@H](CO)C(=O)O",
    "CSCC[C@H](N)C(=O)O",
    "CC(=O)OCc1ccccc1"
]

# --- (옵션) ChEMBL/ZINC 소규모 다운로드 시도 : 실패해도 임베드 데이터로 진행 ---
def try_download_extra(max_n=500):
    """네트워크가 되면 공개 소스에서 SMILES 몇 개를 추가로 시도. 실패 시 빈 리스트."""
    extra = []
    # 예시: ChEMBL REST API 로 소량 조회 (환경/네트워크에 따라 실패할 수 있음)
    try:
        import urllib.request, json
        url = ("https://www.ebi.ac.uk/chembl/api/data/molecule.json"
               "?molecule_properties__num_ro5_violations=0&limit=200")
        with urllib.request.urlopen(url, timeout=15) as r:
            data = json.load(r)
        for m in data.get("molecules", []):
            s = (m.get("molecule_structures") or {}).get("canonical_smiles")
            if s:
                extra.append(s)
            if len(extra) >= max_n:
                break
        print(f"[download] ChEMBL 에서 {len(extra)}개 SMILES 추가 확보")
    except Exception as e:
        print(f"[download] 건너뜀 (오프라인/실패해도 정상): {type(e).__name__}")
    return extra

raw_smiles = list(EMBEDDED_SMILES) + try_download_extra()
print("원시 SMILES 개수:", len(raw_smiles))


In [ ]:
# --- 정제: 파싱 → canonical → 중복 제거 → 길이 필터 ----------------------
MIN_HEAVY, MAX_HEAVY = 5, 40   # 소규모 데모용 heavy-atom 범위

def clean_smiles(smiles_list, min_heavy=MIN_HEAVY, max_heavy=MAX_HEAVY):
    cleaned, seen, n_invalid, n_lenfail = [], set(), 0, 0
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)          # 유효성 검사
        if mol is None:
            n_invalid += 1
            continue
        n_heavy = mol.GetNumHeavyAtoms()
        if not (min_heavy <= n_heavy <= max_heavy):
            n_lenfail += 1
            continue
        can = Chem.MolToSmiles(mol)            # canonical 통일
        if can in seen:
            continue
        seen.add(can)
        cleaned.append(can)
    return cleaned, n_invalid, n_lenfail

train_smiles, n_invalid, n_lenfail = clean_smiles(raw_smiles)
print(f"파싱 실패(무효): {n_invalid} | 길이 필터 제외: {n_lenfail} | 중복 제거 후 최종: {len(train_smiles)}")

# 통계
lengths = [len(s) for s in train_smiles]
heavy   = [Chem.MolFromSmiles(s).GetNumHeavyAtoms() for s in train_smiles]
mw      = [Descriptors.MolWt(Chem.MolFromSmiles(s)) for s in train_smiles]
print(f"SMILES 길이   : min {min(lengths)}, 평균 {np.mean(lengths):.1f}, max {max(lengths)}")
print(f"heavy atoms   : min {min(heavy)}, 평균 {np.mean(heavy):.1f}, max {max(heavy)}")
print(f"분자량(MW)    : min {min(mw):.0f}, 평균 {np.mean(mw):.0f}, max {max(mw):.0f}")
print("\n예시 5개:", train_smiles[:5])


### 문자 vocabulary · 토큰화 · Dataset/DataLoader

SMILES 를 **문자 단위**로 토큰화합니다. 특수 토큰 `PAD`(패딩) · `SOS`(시작) · `EOS`(끝) 를 추가하고,
각 분자를 `SOS + 문자들 + EOS` 로 만들어 배치 내 최대 길이에 맞춰 패딩합니다.


In [ ]:
# --- vocab 구성 ----------------------------------------------------------
PAD, SOS, EOS = "<pad>", "<sos>", "<eos>"
charset = sorted(set("".join(train_smiles)))
itos = [PAD, SOS, EOS] + charset          # index -> token
stoi = {t: i for i, t in enumerate(itos)} # token -> index
PAD_IDX, SOS_IDX, EOS_IDX = stoi[PAD], stoi[SOS], stoi[EOS]
VOCAB = len(itos)
MAX_LEN = max(len(s) for s in train_smiles) + 2   # SOS/EOS 여유
print(f"vocab 크기: {VOCAB} | 최대 시퀀스 길이: {MAX_LEN}")
print("문자 집합:", "".join(charset))

def encode(smi):
    return [SOS_IDX] + [stoi[c] for c in smi] + [EOS_IDX]

def decode(ids):
    toks = []
    for i in ids:
        if i in (SOS_IDX, PAD_IDX):
            continue
        if i == EOS_IDX:
            break
        toks.append(itos[i])
    return "".join(toks)

class SmilesDataset(Dataset):
    def __init__(self, smiles):
        self.data = [encode(s) for s in smiles]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        seq = self.data[idx][:MAX_LEN]
        seq = seq + [PAD_IDX] * (MAX_LEN - len(seq))   # 고정 길이 패딩
        return torch.tensor(seq, dtype=torch.long)

dataset = SmilesDataset(train_smiles)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)
print("배치 예시 shape:", next(iter(loader)).shape)   # (batch, MAX_LEN)


## ② 생성 — char-RNN (문자수준 언어모델)

**LSTM 기반 문자수준 언어모델**을 정의하고, 다음 문자를 예측하도록(teacher forcing) 학습합니다.
소규모 데이터 + 소수 epoch 라 **CPU에서 대략 1~3분** 안에 끝납니다 (개념 확인용).
학습 후 **temperature 샘플링**으로 새 SMILES 를 생성합니다.


In [ ]:
# --- char-RNN 모델 정의 --------------------------------------------------
class CharRNN(nn.Module):
    def __init__(self, vocab, emb=64, hidden=256, layers=2, pad_idx=0):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb, padding_idx=pad_idx)
        self.lstm  = nn.LSTM(emb, hidden, layers, batch_first=True,
                             dropout=0.2 if layers > 1 else 0.0)
        self.fc    = nn.Linear(hidden, vocab)
    def forward(self, x, hidden=None):
        e = self.embed(x)
        out, hidden = self.lstm(e, hidden)
        return self.fc(out), hidden

model = CharRNN(VOCAB, pad_idx=PAD_IDX).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
crit = nn.CrossEntropyLoss(ignore_index=PAD_IDX)   # 패딩은 손실에서 제외
n_params = sum(p.numel() for p in model.parameters())
print(f"파라미터 수: {n_params:,}")


In [ ]:
# --- 학습 루프 (소규모) --------------------------------------------------
EPOCHS = 60   # CPU 데모 기준. 시간이 남으면 늘려보세요.
model.train()
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    total, nb = 0.0, 0
    for batch in loader:
        batch = batch.to(DEVICE)
        inp, tgt = batch[:, :-1], batch[:, 1:]      # 다음 문자 예측
        logits, _ = model(inp)
        loss = crit(logits.reshape(-1, VOCAB), tgt.reshape(-1))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        total += loss.item(); nb += 1
    if epoch % 10 == 0 or epoch == 1:
        print(f"epoch {epoch:3d}/{EPOCHS} | loss {total/nb:.4f} | {time.time()-t0:.1f}s")
print("학습 완료.")


In [ ]:
# --- temperature 샘플링으로 SMILES 생성 ---------------------------------
@torch.no_grad()
def sample_smiles(model, temperature=1.0, max_len=MAX_LEN):
    model.eval()
    x = torch.tensor([[SOS_IDX]], device=DEVICE)
    hidden, out_ids = None, []
    for _ in range(max_len):
        logits, hidden = model(x, hidden)
        logits = logits[:, -1, :] / max(temperature, 1e-6)
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1).item()
        if nxt == EOS_IDX:
            break
        if nxt == PAD_IDX:
            continue
        out_ids.append(nxt)
        x = torch.tensor([[nxt]], device=DEVICE)
    return decode(out_ids)

# 온도(temperature)가 다양성/유효성 균형에 미치는 영향 미리보기
for temp in (0.7, 1.0, 1.2):
    print(f"\n--- temperature = {temp} ---")
    for _ in range(3):
        print(" ", sample_smiles(model, temperature=temp))


In [ ]:
# --- N개 생성 ------------------------------------------------------------
N_GEN = 200
TEMP  = 1.0
generated = [sample_smiles(model, temperature=TEMP) for _ in range(N_GEN)]
generated = [g for g in generated if len(g) > 0]
print(f"{len(generated)}개 생성 (temperature={TEMP})")
print("샘플:", generated[:10])


## (대안) SELFIES 기반 생성 & VAE 개념

- **SELFIES** 는 설계상 **모든 문자열이 유효한 분자로 디코딩**됩니다 (SMILES 의 문법 오류 문제를 제거).
  아래는 데이터에서 뽑은 SELFIES 토큰을 무작위로 이어 붙여 디코딩하는 **최소 데모**로,
  "왜 SELFIES 의 validity 가 1.0 에 가까운가"를 보여줍니다. (별도 학습 없이 개념만 확인)
- **VAE** 는 분자를 연속 잠재공간으로 인코딩→샘플링→디코딩합니다. 아래에 **개념 스니펫**(구조만)을 둡니다.
  char-RNN 이 이 노트북의 **메인 생성기**이고, SELFIES/VAE 는 보조 개념입니다.


In [ ]:
# --- SELFIES 최소 데모: 무작위 토큰 조합도 대부분 유효 -------------------
import selfies as sf

# 데이터셋을 SELFIES 로 변환하고 토큰 알파벳 구성
selfies_list = []
for smi in train_smiles:
    try:
        selfies_list.append(sf.encoder(smi))
    except Exception:
        pass
alphabet = sorted(sf.get_alphabet_from_selfies(selfies_list))
print(f"SELFIES 토큰 종류: {len(alphabet)}")

def random_selfies_molecule(n_tokens=12):
    toks = "".join(random.choice(alphabet) for _ in range(n_tokens))
    smi = sf.decoder(toks)                 # 어떤 조합이든 디코딩 가능
    return smi

rand_smis = [random_selfies_molecule() for _ in range(50)]
valid_rand = [s for s in rand_smis if s and Chem.MolFromSmiles(s) is not None]
print(f"무작위 SELFIES 50개 중 유효: {len(valid_rand)}/50 "
      f"(SMILES 를 무작위로 만들면 대부분 무효인 것과 대조)")
print("예시:", valid_rand[:5])


In [ ]:
# --- (개념 스니펫) 아주 단순한 SMILES VAE 골격 : 참고용, 실행은 선택 ------
# 실제 학습은 생략(개념만). char-RNN 이 메인 생성기입니다.
class TinySmilesVAE(nn.Module):
    """인코더(LSTM)->잠재 z(평균/분산)->디코더(LSTM). 개념 이해용 최소 구조."""
    def __init__(self, vocab, emb=64, hidden=256, latent=32, pad_idx=0):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb, padding_idx=pad_idx)
        self.enc   = nn.LSTM(emb, hidden, batch_first=True)
        self.to_mu = nn.Linear(hidden, latent)
        self.to_lv = nn.Linear(hidden, latent)
        self.z2h   = nn.Linear(latent, hidden)
        self.dec   = nn.LSTM(emb, hidden, batch_first=True)
        self.fc    = nn.Linear(hidden, vocab)
    def encode(self, x):
        _, (h, _) = self.enc(self.embed(x))
        h = h[-1]
        return self.to_mu(h), self.to_lv(h)
    def reparam(self, mu, lv):
        std = torch.exp(0.5 * lv)
        return mu + std * torch.randn_like(std)
    def decode(self, x, z):
        h0 = torch.tanh(self.z2h(z)).unsqueeze(0)
        c0 = torch.zeros_like(h0)
        out, _ = self.dec(self.embed(x), (h0, c0))
        return self.fc(out)
    def forward(self, x):
        mu, lv = self.encode(x)
        z = self.reparam(mu, lv)
        return self.decode(x[:, :-1], z), mu, lv

# VAE 손실 = 재구성 CE + KL 발산. (학습 루프는 char-RNN 과 유사하여 생략)
print("TinySmilesVAE 정의 완료 — 구조 참고용 (학습은 char-RNN 으로 대체).")


## ③ 평가 — validity · uniqueness · novelty

- **Validity**: 생성물 중 RDKit 로 파싱되는 유효 분자 비율
- **Uniqueness**: 유효 분자 중 (중복 제거 후) 서로 다른 분자 비율
- **Novelty**: 유효·고유 분자 중 **학습셋에 없던** 새 분자 비율


In [ ]:
# --- 평가 지표 계산 ------------------------------------------------------
def canonical_or_none(smi):
    m = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(m) if m is not None else None

canon_gen = [canonical_or_none(s) for s in generated]
valid = [c for c in canon_gen if c is not None]
unique = set(valid)
train_set = set(train_smiles)
novel = [c for c in unique if c not in train_set]

n = len(generated)
validity   = len(valid) / n if n else 0.0
uniqueness = len(unique) / len(valid) if valid else 0.0
novelty    = len(novel) / len(unique) if unique else 0.0

print(f"생성 개수      : {n}")
print(f"Validity       : {validity:.3f}  ({len(valid)}/{n})")
print(f"Uniqueness     : {uniqueness:.3f}  ({len(unique)}/{len(valid)})")
print(f"Novelty        : {novelty:.3f}  ({len(novel)}/{len(unique)})")
print("\n(소규모/짧은 학습이라 수치가 낮을 수 있습니다 — 개념 확인이 목적)")


## ④ 필터 — QED · SA · Lipinski (Ro5)

- **QED** (Quantitative Estimate of Drug-likeness): 0~1, 높을수록 drug-like
- **SA score** (Synthetic Accessibility): 1(쉬움)~10(어려움) — RDKit contrib `sascorer`. **추정치**일 뿐 실제 합성경로와 다릅니다.
- **Lipinski Ro5**: MW≤500, HBD≤5, HBA≤10, LogP≤5 위반 개수


In [ ]:
# --- SA scorer 로드 (RDKit contrib) : 실패 시 폴백 ----------------------
sascorer = None
try:
    import sys
    from rdkit.Chem import RDConfig
    sys.path.append(os.path.join(RDConfig.RDContribDir, "SA_Score"))
    import sascorer  # noqa: F401
    print("[ok] sascorer 로드 성공")
except Exception as e:
    print(f"[warn] sascorer 로드 실패 ({type(e).__name__}) — SA 는 NaN 으로 표시됩니다.")

def sa_score(mol):
    if sascorer is None:
        return float("nan")
    try:
        return sascorer.calculateScore(mol)
    except Exception:
        return float("nan")

def ro5_violations(mol):
    v = 0
    if Descriptors.MolWt(mol) > 500: v += 1
    if Lipinski.NumHDonors(mol) > 5: v += 1
    if Lipinski.NumHAcceptors(mol) > 10: v += 1
    if Crippen.MolLogP(mol) > 5: v += 1
    return v


In [ ]:
# --- 후보 프로퍼티 표 만들기 + 랭킹/게이트 ------------------------------
rows = []
for smi in sorted(unique):            # 유효·고유 분자만 대상
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    rows.append({
        "smiles": smi,
        "MW":    round(Descriptors.MolWt(mol), 1),
        "LogP":  round(Crippen.MolLogP(mol), 2),
        "QED":   round(QED.qed(mol), 3),
        "SA":    round(sa_score(mol), 2),
        "Ro5_viol": ro5_violations(mol),
        "novel": smi not in train_set,
    })

df = pd.DataFrame(rows)
print(f"평가 대상 분자: {len(df)}")

if len(df):
    # 게이트: drug-like 기준 (QED 높고, 합성 쉬움 추정, Ro5 위반 없음)
    sa_ok = df["SA"].isna() | (df["SA"] <= 5.0)   # SA 없으면 통과
    gate = (df["QED"] >= 0.5) & sa_ok & (df["Ro5_viol"] == 0)
    passed = df[gate].sort_values("QED", ascending=False)
    print(f"필터 통과(QED>=0.5 & SA<=5 & Ro5 위반 0): {len(passed)}")
    display_cols = ["smiles", "MW", "LogP", "QED", "SA", "Ro5_viol", "novel"]
    print("\n[상위 후보]")
    print(passed[display_cols].head(10).to_string(index=False))
else:
    passed = df
    print("평가할 유효 분자가 없습니다 (학습을 더 돌려보세요).")


## ⑤ 후속 — 시각화 · docking/ADMET 연계 · 조건부 생성

상위 후보를 그림으로 확인하고 QED 분포를 봅니다. docking/ADMET 연계와 조건부 생성은 **개념**으로 정리합니다.


In [ ]:
# --- 상위 후보 구조 그리기 ----------------------------------------------
top = passed if len(passed) else df
top = top.sort_values("QED", ascending=False).head(9) if len(top) else top

if len(top):
    mols = [Chem.MolFromSmiles(s) for s in top["smiles"]]
    legends = [f"QED={q:.2f}" + (f" SA={sa:.1f}" if not pd.isna(sa) else "")
               for q, sa in zip(top["QED"], top["SA"])]
    img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(240, 200), legends=legends)
    display(img)   # Colab/Jupyter 에서 인라인 표시
else:
    print("표시할 후보가 없습니다.")


In [ ]:
# --- QED 분포 히스토그램 -------------------------------------------------
if len(df):
    plt.figure(figsize=(6, 4))
    plt.hist(df["QED"].dropna(), bins=20, color="#2E86C1", edgecolor="white")
    plt.axvline(0.5, color="crimson", linestyle="--", label="게이트 QED=0.5")
    plt.xlabel("QED (drug-likeness)"); plt.ylabel("분자 수")
    plt.title(f"생성 분자 QED 분포 (n={len(df)})")
    plt.legend(); plt.tight_layout(); plt.show()
else:
    print("히스토그램을 그릴 데이터가 없습니다.")


### docking / ADMET 연계 · 조건부 생성 (개념)

- **docking (개념)**: 필터를 통과한 후보를 타깃 단백질 구조에 도킹해 결합 친화도(binding affinity)를 *추정*합니다.
  오픈소스 예로 **AutoDock Vina**, 딥러닝 기반 **DiffDock** 등이 있습니다. docking 점수 역시 **가설**이며 순위 참고용입니다.
- **ADMET (개념)**: 흡수·분포·대사·배설·독성을 in silico 로 예측(예: 용해도, CYP 저해, hERG, 간독성).
  ADMET 예측기로 후보를 추가 스크리닝해 위험 구조를 조기에 걸러냅니다.
- **조건부 생성 (개념)**: "원하는 성질(예: 특정 QED/LogP 범위, 특정 스캐폴드, 타깃 활성)"을 조건으로 주고
  그 조건을 만족하는 분자만 생성하도록 유도합니다. 방법으로는 성질 토큰을 조건으로 주는 conditional LM,
  잠재공간 최적화(VAE + Bayesian optimization), 강화학습(REINVENT류 보상 최적화) 등이 있습니다.

아래는 "생성 → 필터 통과 후보를 docking 입력용 파일로 내보내는" 개념 스니펫입니다 (선택 실행).


In [ ]:
# --- (선택) 필터 통과 후보를 CSV/SDF 로 내보내 docking/ADMET 입력 준비 ---
if len(passed):
    out_csv = "generated_candidates.csv"
    passed.to_csv(out_csv, index=False)
    print(f"[저장] {out_csv} ({len(passed)} 후보)")

    # 3D 좌표 임베딩 후 SDF 로 내보내는 개념 (docking 입력용)
    from rdkit.Chem import AllChem
    writer = Chem.SDWriter("generated_candidates.sdf")
    n_written = 0
    for smi in passed["smiles"].head(5):     # 데모라 상위 5개만
        m = Chem.AddHs(Chem.MolFromSmiles(smi))
        if AllChem.EmbedMolecule(m, randomSeed=SEED) == 0:
            AllChem.MMFFOptimizeMolecule(m)
            writer.write(m); n_written += 1
    writer.close()
    print(f"[저장] generated_candidates.sdf ({n_written} 분자, 3D) — Vina/DiffDock 입력 예시")
else:
    print("내보낼 후보가 없습니다.")


## 마무리 — 한계와 기억할 것

1. **생성 ≠ 합성가능** — SA score 는 *추정치*이며 실제 합성경로는 별개입니다.
2. **in silico 점수(QED·docking·ADMET)는 가설**이지 사실이 아닙니다.
3. **소규모 데모는 성능이 낮습니다** — 목적은 개념 확인. 실전은 대규모 데이터·사전학습·더 큰 모델이 필요합니다.
4. **최종 판단은 실험(합성·assay)** — 모델의 출력은 후보 **'제안'** 까지입니다.

> ⚠️ **함정**: '좋아 보이는 분자'와 '실제로 유용한 분자'는 다릅니다. 생성 결과는 **우선순위 제안**이며,
> 실험 검증 전까지는 결론이 아닙니다.

### 더 해보기
- `EPOCHS` 를 늘리고 데이터를 키워 validity/novelty 변화를 관찰
- `temperature` 를 바꿔 다양성 ↔ 유효성 trade-off 확인
- SELFIES 로 char-RNN 을 재학습해 validity 개선 비교
- 조건부 생성(원하는 QED/LogP 대) 실험

### 참고자료
- **RDKit** — Open-source cheminformatics, https://www.rdkit.org
- **PyTorch** — Paszke et al., *Advances in NeurIPS* 32 (2019), https://pytorch.org
- **SELFIES** — Krenn et al., *Machine Learning: Science and Technology* 1, 045024 (2020)
- **ZINC** — Irwin & Shoichet, *J. Chem. Inf. Model.* 45, 177–182 (2005)
- **ChEMBL** — Mendez et al., *Nucleic Acids Research* 47, D930–D940 (2019)
- **QED** — Bickerton et al., *Nature Chemistry* 4, 90–98 (2012)
- **SA score** — Ertl & Schuffenhauer, *J. Cheminformatics* 1, 8 (2009)
- **벤치마크** — MOSES (Polykovskiy et al., 2020), GuacaMol (Brown et al., 2019)
- 실습 저장소: `github.com/fourmodern/2025_aidrugdiscovery`
